In [1]:
%pip install splink

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     ---------------------------------------- 0.0/55.8 kB ? eta -:--:--
     ---------------------------------------- 55.8/55.8 kB ? eta 0:00:00
   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   ---- ----------------------------------- 0.5/3.7 MB 9.4 MB/s eta 0:00:01
   ------ --------------------------------- 0.6/3.7 MB 9.6 MB/s eta 0:00:01
   ----------- ---------------------------- 1.0/3.7 MB 8.3 MB/s eta 0:00:01
   -------------- ------------------------- 1.3/3.7 MB 7.7 MB/s eta 0:00:01
   -------------- ------------------------- 1.4/3.7 MB 7.1 MB/s eta 0:00:01
   ------------------ --------------------- 1.7/3.7 MB 6.4 MB/s eta 0:00:01
   ------------------ --------------------- 1.8/3.7 MB 5.6 MB/s eta 0:00:01
   ------------------ --------------------- 1.8/3.7 MB 5.6 MB/s eta 0:00:01
   --------------------------- ------------ 2.5/3.7 MB 6.2 MB/s eta 0:00:01


In [13]:
import pandas as pd
from splink.duckdb.linker import DuckDBLinker
from splink.duckdb.blocking_rule_library import block_on
import splink.duckdb.comparison_library as cl
import splink.duckdb.comparison_template_library as ctl

In [14]:
df_l = pd.read_csv('hospital_account_info.csv')
df_r = pd.read_csv('hospital_reimbursement.csv')

In [25]:
from splink.datasets import splink_datasets
df = splink_datasets.fake_1000

# Split a simple dataset into two, separate datasets which can be linked together.
df_l = df.sample(frac=0.5)
df_r = df.drop(df_l.index)

df_l.head(2)

downloading: https://raw.githubusercontent.com/moj-analytical-services/splink_datasets/master/data/fake_1000.csv
  download progress: 100 %	(==========)


,unique_id,first_name,surname,dob,city,email,cluster
74,74,Ronni,Begum,2003-10-15,London,r.b80@ellis-berry.com,22
463,463,Heenry,WilliaWms,2007-02-03,London,henry.w@miller-mitchell.info,117


In [26]:
from splink.duckdb.linker import DuckDBLinker
from splink.duckdb.blocking_rule_library import block_on
import splink.duckdb.comparison_library as cl
import splink.duckdb.comparison_template_library as ctl


settings = {
    "link_type": "link_only",
    "blocking_rules_to_generate_predictions": [
        block_on("first_name"),
        block_on("surname"),
    ],
    "comparisons": [
        ctl.name_comparison("first_name",),
        ctl.name_comparison("surname"),
        ctl.date_comparison("dob", cast_strings_to_date=True),
        cl.exact_match("city", term_frequency_adjustments=True),
        ctl.email_comparison("email", include_username_fuzzy_level=False),
    ],       
}

linker = DuckDBLinker([df_l, df_r], settings, input_table_aliases=["df_left", "df_right"])

In [27]:
linker.completeness_chart(cols=["first_name", "surname", "dob", "city", "email"])

alt.LayerChart(...)

In [28]:
deterministic_rules = [
    "l.first_name = r.first_name and levenshtein(r.dob, l.dob) &lt;= 1",
    "l.surname = r.surname and levenshtein(r.dob, l.dob) &lt;= 1",
    "l.first_name = r.first_name and levenshtein(r.surname, l.surname) &lt;= 2",
    "l.email = r.email"
]

linker.estimate_probability_two_random_records_match(deterministic_rules, recall=0.7)

ParseError: Required keyword: 'this' missing for <class 'sqlglot.expressions.EQ'>. Line 1, Col: 65.
  l.first_name = r.first_name and levenshtein(r.dob, l.dob) &lt;= [4m1[0m

In [29]:
linker.estimate_u_using_random_sampling(max_pairs=1e6, seed=1)

----- Estimating u probabilities using random sampling -----

Estimated u probabilities using random sampling

Your model is not yet fully trained. Missing estimates for:
    - first_name (no m values are trained).
    - surname (no m values are trained).
    - dob (no m values are trained).
    - city (no m values are trained).
    - email (no m values are trained).


In [30]:
session_dob = linker.estimate_parameters_using_expectation_maximisation(block_on("dob"))
session_email = linker.estimate_parameters_using_expectation_maximisation(block_on("email"))
session_first_name = linker.estimate_parameters_using_expectation_maximisation(block_on("first_name"))


----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
l."dob" = r."dob"

Parameter estimates will be made for the following comparison(s):
    - first_name
    - surname
    - city
    - email

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - dob

Iteration 1: Largest change in params was 0.586 in probability_two_random_records_match
Iteration 2: Largest change in params was 0.196 in probability_two_random_records_match
Iteration 3: Largest change in params was 0.086 in the m_probability of first_name, level `All other comparisons`
Iteration 4: Largest change in params was 0.0264 in probability_two_random_records_match
Iteration 5: Largest change in params was 0.0101 in probability_two_random_records_match
Iteration 6: Largest change in params was 0.00444 in probability_two_random_records_match
Iteration 7: Largest change in params was 0.00211 in probability_two_random_r

In [31]:
results = linker.predict(threshold_match_probability=0.9)

In [32]:
results.as_pandas_dataframe(limit=5)

,match_weight,match_probability,source_dataset_l,source_dataset_r,unique_id_l,unique_id_r,first_name_l,first_name_r,gamma_first_name,surname_l,...,dob_l,dob_r,gamma_dob,city_l,city_r,gamma_city,email_l,email_r,gamma_email,match_key
0,3.234488,0.903956,df_left,df_right,57,51,Jayaned,Jayden,1,Bennett,...,2017-01-11,2017-01-11,5,swansea,Swansea,0,8jb88@king.com,None,-1,1
1,3.421364,0.914629,df_left,df_right,925,924,Muhammad,Muhammad,4,Patterson,...,2001-01-26,2000-12-27,3,Bournemouth,Bournehhoutm,0,None,None,-1,0
2,3.421364,0.914629,df_left,df_right,900,903,Aira,Aira,4,Cross,...,1999-01-14,1998-12-15,3,Newastle-upon-Tyne,Newcastle-upon-Tyne,0,a.cross58@browning-nguyen.com,None,-1,0
3,3.421364,0.914629,df_left,df_right,815,820,Logan,Logan,4,Morgan,...,1977-01-29,1976-12-31,3,Coventry,Cooventyr,0,None,loganmorgan43@mcbride-king.com,-1,0
4,3.421364,0.914629,df_left,df_right,37,42,Theodore,Theodore,4,Morris,...,1978-08-19,1978-09-18,3,Birmingham,Birgmhniam,0,t.m39@brooks-sawyer.com,None,-1,0
